# ☁️ Étape 4 — Entraînement Hybride QLoRA (Phase 1 : CPT $\rightarrow$ Phase 2 : SFT) & Export GGUF *(Notebook Google Colab GPU)*

**Environnement** : **C'est le SEUL notebook du projet qui s'exécute sur Google Colab (GPU Tesla T4)** !

**Objectif pédagogique** : Reprendre la base de ton notebook `CameroonianFineTuning.ipynb` et l'améliorer pour entraîner `Mistral-Nemo-Instruct-2407-bnb-4bit` en **deux étapes complémentaires** :
1. **Phase 1 (Continued Pre-training — CPT)** : Immersion linguistique sur les commentaires Camfranglais bruts avec `packing=True`.
2. **Phase 2 (Supervised Fine-Tuning — SFT)** : Apprentissage du dialogue avec le *Chat Template* natif de Mistral et **masquage de la loss sur les instructions (`train_on_responses_only`)**.
3. **Génération des réponses sur `sft_test` & Export GGUF (`q4_k_m`)** vers Hugging Face Hub pour que tu puisses faire toute l'évaluation et l'inférence **en local sur ton PC** à l'Étape 5 !

> 📖 **Important** : Lis `04_hybrid_training_qlora/README.md` pour comprendre les corrections apportées par rapport à `CameroonianFineTuning.ipynb`.

## 1. Installation d'Unsloth sur Colab & Importation Sécurisée du Token Hugging Face

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.4.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from google.colab import userdata
# 🔐 Toujours charger le token depuis les secrets Colab (ne jamais l'écrire en clair dans le code !)
hf_token = userdata.get('hf_token')
print("✅ Token Hugging Face récupéré depuis les secrets Colab.")

---
## 2. Chargement 4-bit du Modèle de Base & Configuration QLoRA (`TODO 1`)

En t'inspirant de ce que tu avais fait dans `CameroonianFineTuning.ipynb`, complète le **`TODO 1`** pour :
1. Charger `unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit` en 4-bit avec `max_seq_length = 2048`.
2. Attacher les adaptateurs LoRA (`r = 16`, `lora_alpha = 16`) sur les 7 modules d'attention et de MLP : `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`.

In [ ]:
# === EXERCICE GUIDÉ : TODO 1 ===
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# TODO 1.1 : Charger `model, tokenizer` avec `FastLanguageModel.from_pretrained(...)`
# sur le modèle "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit".

# --- TON CODE ICI ---

# --------------------

# TODO 1.2 : Configurer PEFT / LoRA avec `FastLanguageModel.get_peft_model(...)`
# en ciblant ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# avec r=16, lora_alpha=16, lora_dropout=0, use_gradient_checkpointing="unsloth".

# --- TON CODE ICI ---

# --------------------

---
## 3. Phase 1 — Continued Pre-Training (CPT) sur le Texte Brut (`TODO 2`)

Lors du **CPT**, nous téléchargeons directement le dataset poussé depuis ton PC à l'Étape 3 (`load_dataset("Cmr237/camfranglais-corpus-cpt")`).
- On ajoute `EOS_TOKEN = tokenizer.eos_token` à la fin de chaque commentaire.
- On active **`packing = True`** dans `SFTConfig`.

In [ ]:
# === EXERCICE GUIDÉ : TODO 2 ===
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# Téléchargement depuis ton dépôt Hugging Face Hub (alimenté par ton PC à l'Étape 3) :
# cpt_dataset = load_dataset("Cmr237/camfranglais-corpus-cpt", split="train", token=hf_token)

EOS_TOKEN = tokenizer.eos_token

def format_cpt_batch(examples):
    # TODO 2.1 : Ajouter `EOS_TOKEN` à la fin de chaque texte de `examples["text"]`
    # --- TON CODE ICI ---
    return {"text": []}
    # --------------------

# TODO 2.2 : Instancier `cpt_trainer = SFTTrainer(...)` avec :
#   - dataset_text_field = "text"
#   - packing = True
#   - per_device_train_batch_size = 2
#   - gradient_accumulation_steps = 4
#   - learning_rate = 1e-4
#   - num_train_epochs = 1
#   - optim = "adamw_8bit"

# --- TON CODE ICI ---
cpt_trainer = None
# --------------------

# cpt_stats = cpt_trainer.train()

---
## 4. Phase 2 — Supervised Fine-Tuning (SFT) avec Chat Template & Masquage de la Loss (`TODO 3`)

Nous passons **le même modèle** en **Phase 2 (SFT)** sur les dialogues (`load_dataset("Cmr237/camfranglais-corpus-sft")`) :
1. Formatage avec `tokenizer.apply_chat_template(...)`.
2. Masquage de la *loss* sur l'instruction avec `train_on_responses_only` (`instruction_part="[INST]"`, `response_part="[/INST]"`).

In [ ]:
# === EXERCICE GUIDÉ : TODO 3 ===
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="mistral")
SYSTEM_PROMPT = "Tu es un ami camerounais chaleureux et drôle. Tu réponds toujours naturellement en camfranglais."

def format_sft_conversations(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        full_convo = [{"role": "system", "content": SYSTEM_PROMPT}] + convo
        # TODO 3.1 : Appliquer `tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False)`
        # --- TON CODE ICI ---
        
        # --------------------
    return {"text": texts}

# TODO 3.2 : Créer `sft_trainer = SFTTrainer(...)` (packing=False, learning_rate=5e-5, per_device_train_batch_size=2)
# et lui appliquer `train_on_responses_only(sft_trainer, instruction_part="[INST]", response_part="[/INST]")`

# --- TON CODE ICI ---
sft_trainer = None
# --------------------

# sft_stats = sft_trainer.train()

---
## 5. Génération des Réponses de Test, Push LoRA & Export GGUF vers Hugging Face (`TODO 4`)

Avant de fermer ta session GPU Colab, nous faisons 3 choses essentielles :
1. Générer les réponses du modèle sur le split `sft_test` et sauvegarder `test_predictions.json` pour que **ton notebook local de l'Étape 5** puisse calculer toutes les métriques sur ton PC sans GPU !
2. Pousser les adaptateurs LoRA sur `Cmr237/Camfranglais`.
3. Exporter et pousser le modèle **GGUF (`q4_k_m`)** (qui était le `# TODO` final de ton ancien notebook) pour que tu puisses discuter avec ton chatbot en local sur ton PC !

In [ ]:
# === EXERCICE GUIDÉ : TODO 4 ===
REPO_ID = "Cmr237/Camfranglais"

# TODO 4.1 : Pousser les adaptateurs LoRA et le tokenizer sur Hugging Face Hub avec `token=hf_token`
# --- TON CODE ICI ---

# --------------------

# TODO 4.2 : Exporter le modèle en GGUF ('q4_k_m') vers Hugging Face Hub pour l'inférence locale sur ton PC
# Astuce : `model.push_to_hub_gguf(REPO_ID, tokenizer, quantization_method="q4_k_m", token=hf_token)`
# --- TON CODE ICI ---

# --------------------
print("🎉 Entraînement GPU et export GGUF terminés ! Retourne sur ton PC en local pour l'Étape 5.")